# Customer Churn Prediction — Model Training & Evaluation

This notebook prepares the Telco customer churn dataset, trains baseline and tree-based models, compares performance, examines class imbalance and classification thresholds, and saves the selected pipeline.

**Evaluation note:** The held-out test set is used for final comparison in this notebook. Threshold exploration on the test set is exploratory; for a production-grade unbiased estimate, select the threshold using validation data or cross-validation, then evaluate once on a final untouched test set.


## 1. Setup and load the dataset

Load the raw dataset and inspect its initial structure.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"D:\project\Customer-Churn-Prediction")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 2. Clean and validate the data

`TotalCharges` is stored as text in the raw CSV. Blank values for customers with zero tenure are treated as zero charges; any other missing values are left for the preprocessing imputer.


In [ ]:

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

print("Missing TotalCharges before treatment:",
      df["TotalCharges"].isna().sum())

zero_tenure_missing = (
    df["TotalCharges"].isna() & (df["tenure"] == 0)
)

print("Missing TotalCharges with zero tenure:",
      zero_tenure_missing.sum())

# Apply the domain-based treatment only to zero-tenure customers
df.loc[zero_tenure_missing, "TotalCharges"] = 0

print("Missing TotalCharges after treatment:",
      df["TotalCharges"].isna().sum())

display(df.loc[zero_tenure_missing, [
    "customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"
]])


In [ ]:

print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values by column:")

display(
    df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Count")
)

print("Target labels:")
display(df["Churn"].value_counts(dropna=False))


## 3. Define features and target

The customer ID is excluded because it is an identifier, not a predictive feature. The target is encoded as `No = 0` and `Yes = 1`.


In [ ]:

TARGET = "Churn"
ID_COLUMN = "customerID"

X = df.drop(columns=[TARGET, ID_COLUMN])
y = df[TARGET].map({"No": 0, "Yes": 1})

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target missing values:", y.isna().sum())

display(X.head())


In [ ]:

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "str", "category", "bool"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

print("\nTotal numerical features:", len(numerical_features))
print("Total categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))


## 4. Train/test split

Use a stratified split so the churn proportion is approximately preserved in both subsets. Keep the test set separate from fitting.


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining churn distribution:")
display(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting churn distribution:")
display(y_test.value_counts(normalize=True).mul(100).round(2))


## 5. Build the preprocessing pipeline

Numerical columns are median-imputed and standardized. Categorical columns are imputed with their most frequent value and one-hot encoded. Unknown categories at prediction time are ignored safely.


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")


In [ ]:

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("Training rows preserved:", X_train_processed.shape[0] == X_train.shape[0])
print("Testing rows preserved:", X_test_processed.shape[0] == X_test.shape[0])


In [ ]:

feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("\nFirst 30 transformed feature names:")

for name in feature_names[:30]:
    print(name)


## 6. Train baseline and tree-based models

The first experiments use the training-fitted preprocessor to create transformed arrays. These models are useful baselines; the cross-validation section below uses full pipelines so preprocessing is refitted inside each fold.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)


In [ ]:
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully.")


In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    random_state=42
)

tree_model.fit(X_train_processed, y_train)

print("Decision Tree trained successfully.")


In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

random_forest_model.fit(X_train_processed, y_train)
gradient_boosting_model.fit(X_train_processed, y_train)

print("Random Forest trained successfully.")
print("Gradient Boosting trained successfully.")


## 7. Holdout-test model comparison

Compare Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting on the same held-out test set.


In [ ]:

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = (
    pd.DataFrame(results)
    .set_index("Model")
    .sort_values("ROC-AUC", ascending=False)
)

display(results_df.round(4))


In [ ]:

metrics_to_plot = [
    "Precision", "Recall", "F1-Score", "ROC-AUC"
]

results_df[metrics_to_plot].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. Classification threshold experiments

The default threshold is 0.50. Lower thresholds generally increase recall while also increasing false positives. These experiments are exploratory because they use the held-out test labels; don't repeatedly optimize against this table and then present it as an unbiased final test result.


In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds_to_test = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for model_name, model in {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}.items():

    probabilities = model.predict_proba(X_test_processed)[:, 1]

    for threshold in thresholds_to_test:
        predictions = (probabilities >= threshold).astype(int)

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1-Score": f1_score(
                y_test, predictions, zero_division=0
            ),
            "Predicted Churners": int(predictions.sum())
        })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(4)
    .sort_values(["Model", "Threshold"])
    .reset_index(drop=True)
)


In [ ]:

plt.figure(figsize=(9, 6))

for model_name in threshold_df["Model"].unique():
    subset = threshold_df[threshold_df["Model"] == model_name]

    plt.plot(
        subset["Threshold"],
        subset["Recall"],
        marker="o",
        label=f"{model_name} — Recall"
    )

    plt.plot(
        subset["Threshold"],
        subset["Precision"],
        marker="x",
        linestyle="--",
        label=f"{model_name} — Precision"
    )

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Precision–Recall Trade-off by Threshold")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

chosen_threshold = 0.35

rf_probabilities = random_forest_model.predict_proba(
    X_test_processed
)[:, 1]

rf_threshold_predictions = (
    rf_probabilities >= chosen_threshold
).astype(int)

print(f"Threshold: {chosen_threshold}")
print(classification_report(
    y_test,
    rf_threshold_predictions,
    target_names=["No Churn", "Churn"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_threshold_predictions,
    display_labels=["No Churn", "Churn"],
    cmap="Blues"
)

plt.title(f"Random Forest — Threshold {chosen_threshold}")
plt.tight_layout()
plt.show()


## 9. Model interpretability

Review the Random Forest feature importances and the largest absolute Logistic Regression coefficients. These are useful signals, but they should be interpreted as model associations rather than causal effects.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    "Feature": preprocessor.get_feature_names_out(),
    "Importance": random_forest_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance", ascending=False
)

display(feature_importance.head(15).round(4))

top_features = feature_importance.head(15).sort_values(
    "Importance", ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 15 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:

logistic_coefficients = pd.DataFrame({
    "Feature": preprocessor.get_feature_names_out(),
    "Coefficient": logistic_model.coef_[0]
})

logistic_coefficients["Absolute_Coefficient"] = (
    logistic_coefficients["Coefficient"].abs()
)

display(
    logistic_coefficients
    .sort_values("Absolute_Coefficient", ascending=False)
    .head(15)
    .round(4)
)


## 10. Cross-validation on the training data

Use stratified five-fold cross-validation. Each pipeline fits its own preprocessor within each fold, helping avoid preprocessing leakage across validation folds.


In [ ]:

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

candidate_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

cv_results = []

for name, estimator in candidate_models.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", estimator)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "roc_auc": "roc_auc"
        },
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-Score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results_df = (
    pd.DataFrame(cv_results)
    .set_index("Model")
)

display(cv_results_df.round(4))


## 11. XGBoost experiments

Compare an unweighted XGBoost model with a class-weighted version. `scale_pos_weight` is calculated from the training labels only.


In [ ]:

from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

xgb_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=2
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_scores = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=1
)

xgb_results = {
    "Accuracy": xgb_scores["test_accuracy"].mean(),
    "Precision": xgb_scores["test_precision"].mean(),
    "Recall": xgb_scores["test_recall"].mean(),
    "F1-Score": xgb_scores["test_f1"].mean(),
    "ROC-AUC": xgb_scores["test_roc_auc"].mean()
}

display(pd.DataFrame([xgb_results], index=["XGBoost"]).round(4))


In [ ]:

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Non-churn customers:", negative_count)
print("Churn customers:", positive_count)
print("scale_pos_weight:", round(scale_pos_weight, 3))


In [ ]:

weighted_xgb_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=2
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

weighted_scores = cross_validate(
    weighted_xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=1
)

weighted_xgb_results = pd.DataFrame([{
    "Accuracy": weighted_scores["test_accuracy"].mean(),
    "Precision": weighted_scores["test_precision"].mean(),
    "Recall": weighted_scores["test_recall"].mean(),
    "F1-Score": weighted_scores["test_f1"].mean(),
    "ROC-AUC": weighted_scores["test_roc_auc"].mean()
}], index=["Weighted XGBoost"])

display(weighted_xgb_results.round(4))


## 12. Weighted XGBoost threshold analysis

Fit the weighted pipeline on the training set, then inspect its holdout predictions at several thresholds. The test-set threshold results are exploratory and should not be used as a final unbiased estimate after threshold selection.


In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Fit only on training data
weighted_xgb_final = clone(weighted_xgb_pipeline)
weighted_xgb_final.fit(X_train, y_train)

# Predict churn probabilities on the untouched test set
y_prob = weighted_xgb_final.predict_proba(X_test)[:, 1]

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Flagged Customers": int(y_pred.sum())
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))

In [ ]:
threshold_df.columns = [
    "Threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC",
    "Flagged Customers"
]

display(
    threshold_df.to_string(
        index=False,
        formatters={
            "Threshold": "{:.2f}".format,
            "Accuracy": "{:.4f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format,
            "ROC-AUC": "{:.4f}".format
        }
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

chosen_threshold = 0.50

y_pred_050 = (y_prob >= chosen_threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_050))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_050,
    target_names=["No Churn", "Churn"]
))

## 13. Fair holdout comparison of the four main candidates

All four pipelines are fitted on the same training split and evaluated on the same test split at their default threshold of 0.50. This comparison is more suitable than comparing results produced by different evaluation procedures.


In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Build pipelines using your existing preprocessing and models
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(logistic_model))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(random_forest_model))
    ]),
    "Original XGBoost": clone(xgb_pipeline),
    "Weighted XGBoost": clone(weighted_xgb_pipeline)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(results).set_index("Model")
display(comparison_df.round(4))

## 14. Train and save the selected model pipeline

The current candidate is Weighted XGBoost because it achieved the highest recall, F1-score, and ROC-AUC in the current holdout comparison. This is a project choice based on the retention objective, not proof that it is universally best.

The exported file includes preprocessing and the estimator together, so inference can start from raw customer feature columns.


In [ ]:
from pathlib import Path
import joblib
from sklearn.base import clone

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "customer_churn_model.pkl"

final_model = clone(weighted_xgb_pipeline)
final_model.fit(X_train, y_train)

joblib.dump(final_model, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")


## 15. Conclusion and next steps

Summarize the final metrics, explain the precision–recall trade-off, and document the model's limitations. Before deployment, validate the chosen threshold on validation data and evaluate once on an untouched test set. The next stage is to build a Streamlit interface that loads `models/customer_churn_model.pkl`.
